# Python drills 5: classes, dataclasses and enums

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-05-classes-and-dataclasses.ipynb)

Companion to the [Classes, dataclasses and enums](../learning/python-05-classes-and-dataclasses.ipynb) chapter. The exercises cover instance state, properties, frozen dataclasses, enums, abstract base classes and equality with hashing.

<details>
<summary><b>How to use this notebook</b></summary>

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

</details>

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

## Exercise 1: a stack class

**Difficulty:** Easy · about 10 min

Write a class `Stack` with these methods:

- `push(item)` adds an item to the top.
- `pop()` removes and returns the top item. An empty stack raises `IndexError`.
- `peek()` returns the top item without removing it.
- `__len__` returns the number of items, so `len(stack)` works.

*Hint: keep the items in a list in `self`, and let the list's own `pop` and `[-1]` do the work.*

In [ ]:
class Stack:
    # your code here
    pass

In [ ]:
check(
    "Stack: pop returns the last pushed item",
    lambda: (lambda s: (s.push(1), s.push(2), s.pop())[-1])(Stack()),
    2,
)
check(
    "Stack: peek and len leave the item in place",
    lambda: (lambda s: (s.push("a"), s.peek(), len(s)))(Stack())[1:],
    ("a", 1),
)
check(
    "Stack: pop on empty raises IndexError",
    lambda: raises(lambda: Stack().pop(), IndexError),
    True,
)
check("Stack: len of a new stack is 0", lambda: len(Stack()), 0)
check(
    "Stack: pops come back in reverse order",
    lambda: (lambda s: (s.push(1), s.push(2), s.push(3), [s.pop(), s.pop(), s.pop()])[-1])(Stack()),
    [3, 2, 1],
)
check(
    "Stack: pop removes the item",
    lambda: (lambda s: (s.push(1), s.push(2), s.pop(), (len(s), s.peek()))[-1])(Stack()),
    (1, 1),
)
check("Stack: peek on empty raises IndexError", lambda: raises(lambda: Stack().peek(), IndexError), True)
check(
    "Stack: two stacks do not share items",
    lambda: (lambda first: (first.push(1), len(Stack()))[-1])(Stack()),
    0,
)
check(
    "Stack: None is a valid item",
    lambda: (lambda s: (s.push(None), s.pop(), len(s))[1:])(Stack()),
    (None, 0),
)
check(
    "Stack: 1000 pushes",
    lambda: (lambda s: ([s.push(i) for i in range(1000)], len(s), s.peek())[1:])(Stack()),
    (1000, 999),
)

<details>
<summary><b>Show solution</b></summary>

```python
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def peek(self):
        return self._items[-1]

    def __len__(self):
        return len(self._items)
```

</details>

## Exercise 2: a temperature with a property and validation

**Difficulty:** Medium · about 10 min

Write a class `Temperature`. Its constructor takes `celsius`. Raise `ValueError` when the value is below absolute zero, which is `-273.15`. Add a read-only property `fahrenheit` that returns `celsius * 9 / 5 + 32`.

In [ ]:
class Temperature:
    # your code here
    pass

In [ ]:
check("Temperature(100).fahrenheit", lambda: Temperature(100).fahrenheit, 212.0)
check("Temperature(0).fahrenheit", lambda: Temperature(0).fahrenheit, 32.0)
check(
    "Temperature(-300) raises ValueError",
    lambda: raises(lambda: Temperature(-300), ValueError),
    True,
)
check("Temperature(-40).fahrenheit == -40.0", lambda: Temperature(-40).fahrenheit, -40.0)
check("Temperature(37.5).fahrenheit == 99.5 (float input)", lambda: Temperature(37.5).fahrenheit, 99.5)
check(
    "Temperature(-273.15) is allowed (exactly absolute zero)",
    lambda: round(Temperature(-273.15).fahrenheit, 2),
    -459.67,
)
check(
    "Temperature(-273.16) raises ValueError",
    lambda: raises(lambda: Temperature(-273.16), ValueError),
    True,
)
check(
    "Temperature.fahrenheit is read-only",
    lambda: raises(lambda: setattr(Temperature(0), "fahrenheit", 1), AttributeError),
    True,
)
check("Temperature.fahrenheit is a property, not a method", lambda: callable(Temperature(0).fahrenheit), False)

<details>
<summary><b>Show solution</b></summary>

```python
class Temperature:
    def __init__(self, celsius):
        if celsius < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32
```

</details>

## Exercise 3: a frozen product with a discount method

**Difficulty:** Medium · about 10 min

Write a frozen dataclass `Product` with two fields, `name` (a string) and `price_cents` (an integer). Add a method `with_discount(percent)` that returns a **new** `Product` whose price is reduced by `percent`, using whole cents with the remainder dropped. The original product must not change.

*Hint: `dataclasses.replace` builds the changed copy, or you can call `Product(...)` directly.*

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Product:
    name: str
    price_cents: int

    def with_discount(self, percent):
        # your code here
        pass

In [ ]:
check(
    "Product.with_discount(10) price",
    lambda: Product("tea", 1000).with_discount(10).price_cents,
    900,
)
check(
    "Product.with_discount leaves the original unchanged",
    lambda: (lambda p: (p.with_discount(10), p.price_cents)[1])(Product("tea", 1000)),
    1000,
)
check(
    "Product is frozen",
    lambda: raises(lambda: setattr(Product("x", 1), "price_cents", 2), AttributeError),
    True,
)
check("Product.with_discount(0) keeps the price", lambda: Product("tea", 1000).with_discount(0).price_cents, 1000)
check("Product.with_discount(100) is free", lambda: Product("tea", 1000).with_discount(100).price_cents, 0)
check("Product.with_discount drops the remainder (995 -> 895)", lambda: Product("tea", 995).with_discount(10).price_cents, 895)
check("Product.with_discount(7) avoids float error (100 -> 93)", lambda: Product("tea", 100).with_discount(7).price_cents, 93)
check("Product.with_discount keeps an int price", lambda: type(Product("tea", 1000).with_discount(10).price_cents), int)
check(
    "Product.with_discount returns a new Product with the same name",
    lambda: (lambda p: (lambda q: (q is not p, type(q), q.name))(p.with_discount(0)))(Product("tea", 1000)),
    (True, Product, "tea"),
)
check(
    "Product.with_discount can be chained",
    lambda: Product("tea", 1000).with_discount(50).with_discount(50).price_cents,
    250,
)

<details>
<summary><b>Show solution</b></summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class Product:
    name: str
    price_cents: int

    def with_discount(self, percent):
        return Product(self.name, self.price_cents * (100 - percent) // 100)
```

</details>

## Exercise 4: an enum and a function that uses it

**Difficulty:** Easy · about 10 min

Define an `Enum` called `Priority` with the members `LOW = 1`, `MEDIUM = 2` and `HIGH = 3`. Then write `highest_name(priorities)`. It takes a list of `Priority` members and returns the **name** of the member with the highest value, as a string.

In [ ]:
from enum import Enum

class Priority(Enum):
    # your members here
    pass

def highest_name(priorities):
    # your code here
    pass

In [ ]:
check(
    "highest_name([LOW, HIGH, MEDIUM]) == 'HIGH'",
    lambda: highest_name([Priority.LOW, Priority.HIGH, Priority.MEDIUM]),
    "HIGH",
)
check("Priority.MEDIUM.value == 2", lambda: Priority.MEDIUM.value, 2)
check("highest_name([LOW]) == 'LOW' (single item)", lambda: highest_name([Priority.LOW]), "LOW")
check("highest_name with duplicates", lambda: highest_name([Priority.MEDIUM, Priority.MEDIUM]), "MEDIUM")
check(
    "highest_name compares values, not names",
    lambda: highest_name([Priority.HIGH, Priority.LOW]),
    "HIGH",
)
check("highest_name returns a str", lambda: type(highest_name([Priority.HIGH])), str)
check("Priority has exactly three members", lambda: [p.name for p in Priority], ["LOW", "MEDIUM", "HIGH"])
check("Priority(3) is Priority.HIGH", lambda: Priority(3) is Priority.HIGH, True)
check(
    "highest_name on 1001 items",
    lambda: highest_name([Priority.LOW] * 1000 + [Priority.MEDIUM]),
    "MEDIUM",
)

<details>
<summary><b>Show solution</b></summary>

```python
from enum import Enum

class Priority(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

def highest_name(priorities):
    return max(priorities, key=lambda priority: priority.value).name
```

</details>

## Exercise 5: a subclass that implements an abstract method

**Difficulty:** Medium · about 10 min

`Exporter` is given below. It is an abstract base class with one abstract method, `export(rows)`. Write `CsvExporter`, a subclass that implements `export`. It joins each row's values with commas, and joins the rows with newlines.

In [ ]:
from abc import ABC, abstractmethod

class Exporter(ABC):
    @abstractmethod
    def export(self, rows):
        """Return the rows as one string."""

class CsvExporter(Exporter):
    # your code here
    pass

In [ ]:
check(
    "CsvExporter().export(...)",
    lambda: CsvExporter().export([["a", "b"], ["c", "d"]]),
    "a,b\nc,d",
)
check(
    "Exporter cannot be created directly",
    lambda: raises(lambda: Exporter(), TypeError),
    True,
)
check("CsvExporter().export([]) == ''", lambda: CsvExporter().export([]), "")
check("CsvExporter with one row", lambda: CsvExporter().export([["a", "b", "c"]]), "a,b,c")
check("CsvExporter has no trailing newline", lambda: CsvExporter().export([["a"], ["b"]]), "a\nb")
check("CsvExporter keeps empty fields", lambda: CsvExporter().export([["", "x"]]), ",x")
check("CsvExporter handles unicode", lambda: CsvExporter().export([["é", "ü"]]), "é,ü")
check("CsvExporter is an Exporter", lambda: isinstance(CsvExporter(), Exporter), True)

<details>
<summary><b>Show solution</b></summary>

```python
from abc import ABC, abstractmethod

class Exporter(ABC):
    @abstractmethod
    def export(self, rows):
        """Return the rows as one string."""

class CsvExporter(Exporter):
    def export(self, rows):
        return "\n".join(",".join(row) for row in rows)
```

</details>

## Exercise 6: equality and hashing for a point

**Difficulty:** Medium · about 15 min

Write a class `Point` with `x` and `y` attributes. Two points are equal when both coordinates match. A point must also be hashable, so it can be a dictionary key. Define both `__eq__` and `__hash__`.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    # your __eq__ and __hash__ here

In [ ]:
check("Point(1, 2) == Point(1, 2)", lambda: Point(1, 2) == Point(1, 2), True)
check("Point(1, 2) != Point(2, 1)", lambda: Point(1, 2) != Point(2, 1), True)
check(
    "Point works as a dictionary key",
    lambda: {Point(1, 2): "a"}[Point(1, 2)],
    "a",
)
check("Point(1, 2) != Point(1, 3) (y must match)", lambda: Point(1, 2) != Point(1, 3), True)
check("Point(1, 2) != Point(5, 2) (x must match)", lambda: Point(1, 2) != Point(5, 2), True)
check("Point(0, 0) == Point(0, 0)", lambda: Point(0, 0) == Point(0, 0), True)
check("Point(-1, 0) != Point(1, 0)", lambda: Point(-1, 0) != Point(1, 0), True)
check("equal points have equal hashes", lambda: hash(Point(1, 2)) == hash(Point(1, 2)), True)
check(
    "a set of points removes duplicates",
    lambda: len({Point(1, 2), Point(1, 2), Point(2, 1)}),
    2,
)
check("Point(1, 2) == (1, 2) is False", lambda: Point(1, 2) == (1, 2), False)

<details>
<summary><b>Show solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))
```

</details>

## Score

In [ ]:
score()